> Experiments with [model.signal](../trapallada/model/signal.py)  module

In [1]:
import pathlib

import yaml

import jax.random as jr

import trapallada.data
import trapallada.model.nn
import trapallada.util
import trapallada.core

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Directory containing the data

In [3]:
# input_directory = pathlib.Path.cwd().parent / 'samples' / 'data' / 'mini_rate'
input_directory = pathlib.Path.cwd().parent / 'data' / 'rate_backv'
assert input_directory.exists()
input_directory

PosixPath('/home/manu/Sync/git/trapallada/data/rate_backv')

Data is stored in two separate files:

- data itself, in the form of an `Array`

- metadata in a *Pandas* `DataFrame`

In [4]:
signals_file = input_directory / 'signals.pt'
assert signals_file.exists()

masks_file = input_directory / 'masks.pt'
assert masks_file.exists()

metadata_file = input_directory / 'metadata.parquet'
assert metadata_file.exists

It it turned into a *PyTorch* `Dataset`

In [5]:
ds = trapallada.data.SignalsDataset(signals_file, metadata_file, masks_file, normalize=True)
len(ds), ds[0][1].shape

trapallada.data - INFO - normalizing signals...


(44658, torch.Size([2, 225]))

A parameters file that specifies an experiment

In [6]:
parameters_file = pathlib.Path.cwd().parent / 'parameters.yaml'
# parameters_file = pathlib.Path.cwd().parent / 'samples' / 'fft_vanilla_mini.yaml'
assert parameters_file.exists()

It is read into a `dic`

In [7]:
with parameters_file.open() as f:
    parameters = yaml.safe_load(f)

In [8]:
%pycat {parameters_file}

PRNG seeds:
  model initialization:
    decoder: 43
    encoder: 42
  sampling:
    training set: 14
    validation set: 15
  training dataloader: 43
  training-validation splitting: 42
loss:
  x_sd: 0.15 # <---- FFT
model: &model
  channels:
  - 12
  - 12
  - 12
  dim_z: 50
  kernel_sizes: # <---- FFT (due to redundancy)
  - 64
  - 32
  - 16
  stride: 2
data:
    normalize: false
    fft: false # <---- FFT
training:
  batch_size: 32
  dl_rng_key: 43
  learning_rate: 3.0e-05
  n_epochs: 2
  train_test_split:
    test_size: 0.2
VAE:
  encoder:
    class: Encoder
    parameters:
      << : [*model]
  decoder:
    class: Decoder # <---- FFT (NonNegativeDecoder)
    parameters:
      << : [*model]


`DataLoaders` are derived from the `Dataset`

In [9]:
train_dl, validation_dl = trapallada.data.dataloaders_from_dataset(
    ds,
    parameters['training']['train_test_split'] | dict(random_state=parameters['PRNG seeds']['training-validation splitting']),
    parameters['training']['batch_size'], parameters['PRNG seeds']['training dataloader'])


How many samples per batch?

In [10]:
batch = next(iter(train_dl))
len(batch)

4

The *index component* in the first batch

In [11]:
batch[0]

tensor([44309, 40815,  3156,  7803, 34442, 17579, 43283, 14708, 30631, 40426,
         8585, 37349, 29793, 10889, 26570, 31776, 22002, 14241, 10663,  6153,
         5647, 32968, 11765, 23928, 21350, 29812,  4698, 23817, 14025, 27387,
         9104, 37849])

The *x component* in the 1st element of the `Dataset`

In [12]:
signals_len = ds[0][1].shape
signals_len

torch.Size([2, 225])

What are the dimensions of every input?

In [13]:
tuple(ds[0][1].shape)

(2, 225)

Both *encoder* and *decoder* also require the *input size* as a parameter: it's inferred from the `Dataset` and included in the corresponding `dict`s

In [14]:
input_size = list(ds[0][1].shape)
for m in ['encoder', 'decoder']:
        parameters['VAE'][m]['parameters']['input_size'] = input_size

In [15]:
x = batch[1][0].numpy()
encoder = trapallada.model.nn.Encoder(
    **parameters['VAE']['encoder']['parameters'],
    key=jr.PRNGKey(parameters['PRNG seeds']['model initialization']['encoder']))
encoder.debug_call(x)
print(encoder)

after equinox.nn._conv: (12, 81)
after jax._src.nn.functions: (12, 81)
after equinox.nn._normalisation: (12, 81)
after equinox.nn._conv: (12, 25)
after jax._src.nn.functions: (12, 25)
after equinox.nn._normalisation: (12, 25)
after equinox.nn._conv: (12, 5)
after jax._src.nn.functions: (12, 5)
after equinox.nn._normalisation: (12, 5)
after jax._src.numpy.lax_numpy: (60,)
after equinox.nn._linear: (100,)
after trapallada.model.nn: (100,)
Encoder(
  layers=[
    Conv1d(
      num_spatial_dims=1,
      weight=f32[12,2,64],
      bias=f32[12,1],
      in_channels=2,
      out_channels=12,
      kernel_size=(64,),
      stride=(2,),
      padding=((0, 0),),
      dilation=(1,),
      groups=1,
      use_bias=True
    ),
    <wrapped function relu>,
    LayerNorm(
      shape=(12, 81),
      eps=1e-05,
      use_weight=True,
      use_bias=True,
      weight=f32[12,81],
      bias=f32[12,81]
    ),
    Conv1d(
      num_spatial_dims=1,
      weight=f32[12,12,32],
      bias=f32[12,1],
      

In [16]:
x.shape

(2, 225)

In [17]:
import equinox as eqx
eqx.nn.Conv1d(in_channels=x.shape[0], out_channels=1, kernel_size=3, stride=1, padding=1, key=jr.PRNGKey(42))(x).shape

(1, 225)

In [18]:
encoder.layers_spatial_sizes

[225, 81, 25, 5, 60]

In [19]:
z = encoder(x)
z.shape

(100,)

In [20]:
z_mean = jr.normal(jr.PRNGKey(42), (parameters['model']['dim_z'],))
decoder = trapallada.model.nn.Decoder(
    **(parameters['VAE']['decoder']['parameters'] | {'layers_spatial_sizes': encoder.layers_spatial_sizes}),
    key=jr.PRNGKey(parameters['PRNG seeds']['model initialization']['decoder']))
decoder.debug_call(z_mean)

after equinox.nn._linear: (60,)
after jax._src.nn.functions: (60,)
after trapallada.model.nn: (12, 5)
after equinox.nn._conv: (12, 25)
after jax._src.nn.functions: (12, 25)
after equinox.nn._normalisation: (12, 25)
after equinox.nn._conv: (12, 81)
after jax._src.nn.functions: (12, 81)
after equinox.nn._normalisation: (12, 81)
after equinox.nn._conv: (2, 225)


In [21]:
vae = trapallada.model.nn.VAE(encoder, decoder)
x_pred, z_mean, z_sd = vae(x, jr.PRNGKey(42))
x_pred.shape, z_mean.shape, z_sd.shape

((2, 225), (50,), (50,))

In [22]:
model, results_df = trapallada.core.fit_vae_on_signals(train_dl, validation_dl, parameters)

    training                         validation                        
        loss loglikelihood        kl       loss loglikelihood        kl
0  36.564955    -36.480439  0.084517  35.475302    -35.389332  0.085971
    training                         validation                        
        loss loglikelihood        kl       loss loglikelihood        kl
1  34.042939    -33.942044  0.100895  32.874326    -32.756883  0.117443


# Conditional VAE

In [23]:
cond_decoder = trapallada.model.nn.ConditionalDecoder(
    **(parameters['VAE']['decoder']['parameters'] | {'input_size': [1, input_size[1]]}| {'layers_spatial_sizes': encoder.layers_spatial_sizes}),
    key=jr.PRNGKey(parameters['PRNG seeds']['model initialization']['decoder']))
# cond_decoder = trapallada.model.nn.ConditionalDecoder(decoder, key=jr.PRNGKey(parameters['PRNG seeds']['model initialization']['decoder']))
# cond_decoder.z_decoder.layers[0].weight.shape
# cond_decoder.z_decoder.layers[-1].out_channels

In [24]:
cond_decoder(z_mean, x[-1:])
# x.shape, z.shape

Array([[-1.27686840e-03,  2.14865729e-01,  2.92266637e-01,
         1.80078730e-01,  2.01041579e-01,  8.02402496e-02,
         9.06054378e-02,  1.39552847e-01,  2.58846939e-01,
         1.51057929e-01,  3.48262787e-01,  1.48237735e-01,
         1.19392216e-01,  1.18993014e-01,  2.20863536e-01,
         9.18692797e-02,  9.07534808e-02,  1.51157483e-01,
         6.15778640e-02, -7.33273849e-02, -4.95893598e-01,
        -6.61169410e-01, -8.30662966e-01, -3.20940614e-01,
        -1.16349220e-01, -2.75283128e-01, -3.62739295e-01,
        -1.29878014e-01,  1.02377519e-01,  9.68106389e-02,
         2.93613672e-01,  1.68018475e-01,  3.80235314e-01,
         1.93761081e-01,  5.46100736e-01,  1.20242089e-01,
        -9.07314301e-01, -1.34636962e+00, -8.22518587e-01,
         1.04116291e-01,  4.40284967e-01,  7.19001949e-01,
         6.88879371e-01,  1.02263451e-01, -3.19172591e-01,
        -3.09082419e-01,  2.30353981e-01,  2.05727220e-01,
         4.35166180e-01,  6.38101697e-01,  3.56925786e-0

In [25]:
cvae = trapallada.model.nn.CVAE(encoder, cond_decoder)
x_pred, z_mean, z_sd = cvae(x, jr.PRNGKey(42))
x_pred.shape, z_mean.shape, z_sd.shape

((1, 225), (50,), (50,))